In [1]:
import os
import sys

In [3]:
sys.path.append(os.path.abspath("../.."))

In [4]:
from datalemur_utils import extract_iframe_from_url

In [5]:
url = "https://datalemur.com/questions/odd-even-measurements"
extract_iframe_from_url(url)

In [6]:
from pyspark.sql import SparkSession

In [7]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/10 00:38:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
measurements = spark.read\
.option("header", "true")\
.option("delimiter", "\t")\
.option("inferSchema", "true")\
.format("csv")\
.load("measurements.txt")

In [10]:
measurements.show()

+--------------+-----------------+-------------------+
|measurement_id|measurement_value|   measurement_time|
+--------------+-----------------+-------------------+
|        131233|          1109.51|07/10/2022 09:00:00|
|        135211|          1662.74|07/10/2022 11:00:00|
|        523542|          1246.24|07/10/2022 13:15:00|
|        143562|           1124.5|07/11/2022 15:00:00|
|        346462|          1234.14|07/11/2022 16:45:00|
+--------------+-----------------+-------------------+



In [11]:
measurements.createOrReplaceTempView("measurements")

In [17]:
spark.sql(
    """
    with ranked_measures as
    (
    select to_date(measurement_time, 'MM/dd/yyyy HH:mm:ss') as measurement_date,
    measurement_value,
    row_number() over(
    partition by to_date(measurement_time, 'MM/dd/yyyy HH:mm:ss')
    order by measurement_time
    ) as measure_num
    from
    measurements
    )
    select
    measurement_date,
    sum(case when measure_num%2 != 0 then measurement_value end) as odd_sum,
    sum(case when measure_num%2 = 0 then measurement_value end) as even_sum
    from
    ranked_measures
    group by 1
    """
)\
.show()

+----------------+-------+--------+
|measurement_date|odd_sum|even_sum|
+----------------+-------+--------+
|      2022-07-10|2355.75| 1662.74|
|      2022-07-11| 1124.5| 1234.14|
+----------------+-------+--------+

